# RL anti-sway crane: reproduce on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zephyr-sue/rl_crane_antisway/blob/master/notebooks/colab_reproduce.ipynb)

This notebook reproduces the **GRPO vs PPO vs RBF adaptive control** benchmark in four steps:
1. Clone the repo and install the dependencies (Colab already ships CUDA PyTorch).
2. Run the tests.
3. Evaluate the **committed trained agents** and regenerate every figure.
4. Optionally retrain GRPO on the Colab GPU.

Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!git clone https://github.com/Zephyr-sue/rl_crane_antisway.git
%cd rl_crane_antisway
!pip install -q gymnasium stable-baselines3 pygame plotly tensorboard pytest
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# Physics (energy conservation, gust work), CPU/GPU simulator equivalence, env API, GRPO advantages, RBF controller
!python -m pytest -q

In [ ]:
# Benchmark the committed agents. Colab has 2 CPU cores, so use fewer episodes for speed (the paper uses 100).
!python evaluate.py --episodes 30 --heatmap-episodes 4 --workers 2 --skip-ablations
import pandas as pd
pd.read_csv('results/summary.csv').pivot(index='condition', columns='controller', values='success').round(2)

In [ ]:
from IPython.display import Image, display
for f in ['robustness_heatmap.png', 'trajectories.png', 'learning_curves.png']:
    display(Image(f'results/{f}', width=700))

In [ ]:
# Side-by-side animation of all controllers on one gusty scenario
!python viz/animate.py --condition "Strong gusts" --index 2 --out results/compare_colab.gif
display(Image('results/compare_colab.gif'))

## Optional: retrain GRPO on the Colab GPU

GRPO simulates 64 scenarios × 32 rollouts (2048 cranes) in parallel on the GPU. A short run of 40 iterations (about 50M environment steps) already gets past 85% validation success. The full run uses 250 iterations.

PPO is CPU-bound (parallel simulator processes), so it is much slower on Colab's 2 cores; train it locally.

In [ ]:
!python train.py --iterations 40 --seed 0 --out models/trained/colab_grpo.pt --tag colab_